##### Purpose

Register the neural-network candidate selected in 07_mlflow_experiment_comparison into the Unity Catalog Model Registry.

The workflow is:

``` text

07_mlflow_experiment_comparison
        ↓
Selected Candidate
        ↓
Exact Logged Model URI
        ↓
Verify Model Artifact
        ↓
Register in Unity Catalog
        ↓
Registered Model Version
        ↓
Candidate Alias
        ↓
Verify Registered Model
        ↓
09_final_testing_and_conclusion

```

##### Production Design

Responsibilities remain separated:

07_mlflow_experiment_comparison
- train
- compare
- select
- persist model

08_model_registration
- verify selected artifact
- register exact artifact
- version it
- assign lifecycle alias
- preserve lineage
 
09_final_testing_and_conclusion
- load Candidate
- final testing
- acceptance decision
- optionally promote to Champion

Notebook 08 therefore performs governance, not training.

##### Technologies Used

- Databricks
- MLflow 3
- Unity Catalog
- MLflow Model Registry
- PyTorch
- Python

##### Input

The notebook needs the selected values produced by Notebook 07:

- SELECTED_EXPERIMENT
- SELECTED_RUN_ID
- SELECTED_MODEL_URI
- SELECTED_VALIDATION_LOSS
- SELECTED_BEST_EPOCH

##### 1. Load Project Configuration

In [0]:
%run ./00_project_config

##### 2. Import Required Components

In [0]:
import mlflow
import torch

from mlflow import MlflowClient

##### 3. Configure Unity Catalog Registry

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

##### 4. Bring Forward the Selected Candidate

In [0]:
import json

with open(
    SELECTION_MANIFEST_PATH,
    "r",
) as file:

    selection_manifest = json.load(
        file
    )

In [0]:
selection_manifest

In [0]:
SELECTED_EXPERIMENT = (
    selection_manifest[
        "selected_experiment"
    ]
)

SELECTED_RUN_ID = (
    selection_manifest[
        "selected_run_id"
    ]
)

SELECTED_MODEL_URI = (
    selection_manifest[
        "selected_model_uri"
    ]
)

SELECTED_VALIDATION_LOSS = (
    selection_manifest[
        "best_validation_loss"
    ]
)

SELECTED_BEST_EPOCH = (
    selection_manifest[
        "best_epoch"
    ]
)

In [0]:
print(
    "Selected experiment:",
    SELECTED_EXPERIMENT,
)

print(
    "Selected Run ID:",
    SELECTED_RUN_ID,
)

print(
    "Selected Model URI:",
    SELECTED_MODEL_URI,
)

print(
    "Best validation loss:",
    SELECTED_VALIDATION_LOSS,
)

print(
    "Best epoch:",
    SELECTED_BEST_EPOCH,
)

##### 5. Validate Required Candidate Metadata

In [0]:
required_candidate_values = {
    "SELECTED_EXPERIMENT":
        SELECTED_EXPERIMENT,

    "SELECTED_RUN_ID":
        SELECTED_RUN_ID,

    "SELECTED_MODEL_URI":
        SELECTED_MODEL_URI,

    "SELECTED_VALIDATION_LOSS":
        SELECTED_VALIDATION_LOSS,

    "SELECTED_BEST_EPOCH":
        SELECTED_BEST_EPOCH,
}


missing_values = [
    name
    for name, value
    in required_candidate_values.items()
    if value is None
]


if missing_values:

    raise ValueError(
        "Missing candidate metadata: "
        f"{missing_values}"
    )

##### 6. Verify the Selected Logged Model Can Be Loaded

In [0]:
selected_model = (
    mlflow.pytorch.load_model(
        SELECTED_MODEL_URI
    )
)

selected_model.eval()

print(
    selected_model
)

##### 7. Inspect Selected Model Parameters

In [0]:
for name, parameter in (
    selected_model.named_parameters()
):

    print(
        name,
        parameter.shape,
    )

##### 8. Register the Exact Logged Model

In [0]:
registered_model_version = (
    mlflow.register_model(
        model_uri=SELECTED_MODEL_URI,
        name=REGISTERED_MODEL_NAME,
    )
)

##### 9. Capture Registered Version

In [0]:
MODEL_VERSION = str(
    registered_model_version.version
)

In [0]:
print(
    "Registered model:",
    REGISTERED_MODEL_NAME,
)

print(
    "Registered version:",
    MODEL_VERSION,
)

print(
    "Source model URI:",
    SELECTED_MODEL_URI,
)

print(
    "Source Run ID:",
    SELECTED_RUN_ID,
)

##### 10. Create MLflow Registry Client

In [0]:
client = MlflowClient()

##### 11. Add Registered Model Description

In [0]:
client.update_registered_model(
    name=REGISTERED_MODEL_NAME,
    description=(
        "PyTorch neural network for IBM Telco "
        "Customer Churn prediction. "
        "Candidate versions are selected through "
        "controlled validation experiments tracked "
        "with MLflow."
    ),
)

##### 12. Add Model Version Description

In [0]:
client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=MODEL_VERSION,
    description=(
        f"Selected from experiment "
        f"'{SELECTED_EXPERIMENT}'. "
        f"Source run: {SELECTED_RUN_ID}. "
        f"Best epoch: {SELECTED_BEST_EPOCH}. "
        f"Best validation loss: "
        f"{SELECTED_VALIDATION_LOSS:.6f}."
    ),
)

##### 13. Add Version Tags

In [0]:
client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=MODEL_VERSION,
    key="source_run_id",
    value=SELECTED_RUN_ID,
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=MODEL_VERSION,
    key="source_experiment",
    value=SELECTED_EXPERIMENT,
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=MODEL_VERSION,
    key="selection_metric",
    value="best_validation_loss",
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=MODEL_VERSION,
    key="lifecycle_status",
    value="candidate",
)

##### 14. Assign Candidate Alias

In [0]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=CANDIDATE_ALIAS,
    version=MODEL_VERSION,
)

##### 15. Verify the Alias

In [0]:
import mlflow  
client = mlflow.tracking.MlflowClient()


candidate_version = (

    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CANDIDATE_ALIAS,
    )
)

In [0]:
print(
    "Alias:",
    CANDIDATE_ALIAS,
)

print(
    "Alias points to version:",
    candidate_version.version,
)

print(
    "Registered model:",
    candidate_version.name,
)

##### 16. Build Candidate Model URI

In [0]:
CANDIDATE_MODEL_URI = (
    f"models:/"
    f"{REGISTERED_MODEL_NAME}"
    f"@{CANDIDATE_ALIAS}"
)

In [0]:
print(
    "Candidate Model URI:",
    CANDIDATE_MODEL_URI,
)

#####17. Load Registered Candidate Through Unity Catalog

In [0]:
registered_candidate_model = (
    mlflow.pytorch.load_model(
        CANDIDATE_MODEL_URI
    )
)

registered_candidate_model.eval()

print(
    registered_candidate_model
)

##### 18. Verify Parameter Keys Match

SELECTED_MODEL_URI
→ identifies the stored model

mlflow.pytorch.load_model()
→ knows how to deserialize/load that stored model
   as a PyTorch model

In [0]:
SELECTED_MODEL_URI

In [0]:
selected_model = (
    mlflow.pytorch.load_model(
        SELECTED_MODEL_URI
    )
)

selected_model.eval()

In [0]:
source_state = (
    selected_model.state_dict()
)

registered_state = (
    registered_candidate_model.state_dict()
)

In [0]:
print(
    "Same parameter keys:",
    source_state.keys()
    == registered_state.keys(),
)

##### 19. Verify Every Weight and Bias Matches

In [0]:
all_parameters_match = all(
    torch.equal(
        source_state[key],
        registered_state[key],
    )
    for key in source_state
)

print(
    "All registered parameters match source:",
    all_parameters_match,
)

##### 20. Verify Version Metadata

In [0]:
registered_version_info = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=MODEL_VERSION,
    )
)

In [0]:
print(
    "Model name:",
    registered_version_info.name,
)

print(
    "Version:",
    registered_version_info.version,
)

print(
    "Run ID:",
    registered_version_info.run_id,
)

print(
    "Description:",
    registered_version_info.description,
)

##### 21. Build Registration Summary

In [0]:
registration_summary = {
    "registered_model_name":
        REGISTERED_MODEL_NAME,

    "registered_model_version":
        MODEL_VERSION,

    "alias":
        CANDIDATE_ALIAS,

    "source_experiment":
        SELECTED_EXPERIMENT,

    "source_run_id":
        SELECTED_RUN_ID,

    "source_model_uri":
        SELECTED_MODEL_URI,

    "best_epoch":
        SELECTED_BEST_EPOCH,

    "best_validation_loss":
        SELECTED_VALIDATION_LOSS,

    "candidate_model_uri":
        CANDIDATE_MODEL_URI,

    "parameters_verified":
        all_parameters_match,
}

In [0]:
for key, value in (
    registration_summary.items()
):

    print(
        f"{key}: {value}"
    )

##### 22. Understand the Lifecycle Transition

BEFORE REGISTRATION

``` text

MLflow Experiment
      ↓
Run
      ↓
Logged Model
models:/m-...

```

AFTER REGISTRATION

``` text

Unity Catalog
      ↓
Registered Model
      ↓
Version
      ↓
Candidate Alias
      ↓
models:/catalog.schema.model@Candidate

```

The neural-network parameters haven't changed.

Only its lifecycle status and governance changed.

##### 23. Run ID vs Logged Model ID vs Version vs Alias

Run ID
↓
Which training experiment produced this?


Logged Model URI
models:/m-...
↓
Which exact MLflow model artifact?


Registered Version
Version 1 / 2 / ...
↓
Which governed version in Unity Catalog?


Alias
Candidate
↓
Which version currently has this lifecycle role?

##### 24. Why Candidate, Not Champion?

At this stage:

``` text

Experiment comparison
       ↓
Selected candidate
       ↓
Registered
       ↓
Candidate

```

We still need Notebook 09 to validate the registered artifact itself.

Only after final acceptance testing should we consider:

``` text

Candidate
    ↓
PASS
    ↓
Champion

```

This is safer than promoting immediately after experiment selection.

##### Key Learnings

1. Model registration is separate from model training.

2. Notebook 08 does not retrain the selected neural network.

3. The exact MLflow Logged Model URI selected in Notebook 07 is registered.

4. MLflow 3 Logged Models can be referenced using models:/<model_id> URIs.

5. Models in Unity Catalog use three-level names:

   catalog.schema.model

6. Model Registry creates governed, versioned model assets.

7. Registering a new candidate creates a new version rather than replacing older versions.

8. A Run ID identifies the training experiment.

9. A Logged Model URI identifies the exact tracked model artifact.

10. A registered model version identifies the governed Unity Catalog version.

11. A model alias provides a movable lifecycle reference.

12. Candidate identifies the version currently awaiting final acceptance.

13. Champion can later identify the approved production version.

14. Registration does not change the neural-network architecture, weights, or biases.

15. Comparing state_dict values verifies that the registered model is identical to the selected source model.

16. Model descriptions and tags improve lineage and governance.

17. Downstream systems should prefer lifecycle aliases over hard-coded model-version numbers when appropriate.

18. Unity Catalog provides centralized model governance, access control, lineage, versioning, auditing, and discovery.

19. Model registration is the transition from experiment tracking to controlled model lifecycle management.

20. Final acceptance testing should validate the registered Candidate before promotion.

##### Conclusion

This notebook registered the neural-network candidate selected in the MLflow experiment-comparison workflow.

The selected MLflow Logged Model artifact was loaded and verified before registration.

The exact artifact was then registered in the Unity Catalog Model Registry.

Registration created a governed model version without retraining or modifying the neural network.

The registered version was enriched with:

- source Run ID,
- source experiment,
- validation-selection information,
- description,
- lifecycle tags,
- and a Candidate alias.

The Candidate model was successfully loaded back from Unity Catalog.

Its PyTorch parameter keys, weights, and biases were verified against the original selected MLflow artifact.

The registered model is now ready for final acceptance testing.

##### Next Notebook

##### 09_final_testing_and_conclusion

The final notebook will test the registered Candidate rather than relying on the training notebook or MLflow experiment artifact.

Workflow:

``` text

Candidate Alias
      ↓
Load Registered Model
      ↓
Load Fitted Preprocessing Artifact
      ↓
Prepare Controlled Test Inputs
      ↓
Run Inference
      ↓
Validate Logits
      ↓
Validate Probabilities
      ↓
Validate Predictions
      ↓
Calculate Final Metrics
      ↓
Compare Against Acceptance Criteria
      ↓
PASS / FAIL
      ↓
Promote Candidate → Champion if approved
      ↓
Final Deep Learning Project Conclusion

```